# Iceberg из чистого Python (pyiceberg)

Spark — не единственная дверь в Iceberg. Библиотека `pyiceberg` ходит в тот же
Hive Metastore и читает/пишет те же таблицы без JVM вообще: подойдёт для лёгких
скриптов, дашбордов, да и просто чтобы понять формат руками.

Запусти сначала `make spark-demo`, чтобы таблица `analytics.sales_by_region_product` существовала.

In [ ]:
!pip install -q "pyiceberg[hive,pandas]"

In [ ]:
from pyiceberg.catalog import load_catalog

catalog = load_catalog("hms", **{
    "type": "hive",
    "uri": "thrift://hive-metastore:9083",
    "s3.endpoint": "http://minio:9000",
    "s3.access-key-id": "minioadmin",
    "s3.secret-access-key": "minioadmin",
})
print("namespaces:", catalog.list_namespaces())
print("tables:    ", catalog.list_tables("analytics"))

## Чтение: scan → pandas

`scan()` читает parquet-файлы напрямую из MinIO (метастор только говорит, где они лежат).

In [ ]:
table = catalog.load_table("analytics.sales_by_region_product")
print(table.schema())
table.scan().to_pandas()

## Фильтры и проекции — pushdown

Фильтр уходит в движок: pyiceberg по статистике в метаданных отбрасывает целые
файлы, не читая их. На больших таблицах это и есть главная магия формата.

In [ ]:
table.scan(
    row_filter="region == 'EMEA'",
    selected_fields=("region", "total_amount"),
).to_pandas()

## Метаданные: снапшоты и файлы

То же, что `hive.analytics.sales_by_region_product.snapshots` в Spark, но без Spark.

In [ ]:
print(table.inspect.snapshots().to_pandas()[["committed_at", "operation", "snapshot_id"]])
print(table.inspect.files().to_pandas()[["file_path", "record_count", "file_size_in_bytes"]])

## Запись из Python — append

Да, pyiceberg умеет и писать: подсунь pyarrow-таблицу с той же схемой.
Новый снапшот увидят и Spark, и Impala (каталог-то общий).

In [ ]:
import pyarrow as pa
from decimal import Decimal

row = pa.table({
    "region": ["TEST"],
    "product": ["from-python"],
    "total_amount": pa.array([Decimal("1")], type=pa.decimal128(38, 18)),
})
table.append(row)
print("строк стало:", len(table.scan().to_pandas()))

## Удаление — и прибраться за собой

`delete()` в pyiceberg работает как copy-on-write: переписывает файлы без
position-delete'ов, так что ClickHouse `icebergS3()` таблицу читать не перестанет
(в отличие от DELETE из Impala — см. ноутбук 03).

In [ ]:
table.delete(delete_filter="region == 'TEST'")
print("строк после удаления:", len(table.scan().to_pandas()))

## Когда pyiceberg, а когда Spark

- **pyiceberg** — лёгкое чтение, мелкие append'ы, метаданные, интеграция в обычный питон-код.
- **Spark** — тяжёлые трансформации, MERGE, большие объёмы, процедуры обслуживания
  (компакция, expire_snapshots — см. ноутбук 05).